3.1 Precipation data

In [2]:
import pandas as pd

# Exploration of the data's encoding:
!curl -s "https://opendata.dwd.de/climate_environment/CDC/helprr_wetter_tageswerte_Beschreibung_Stationen.txt" | file -

/dev/stdin: ISO-8859 text, with very long lines (1000), with CRLF line terminators


In [32]:
# 1.
# DATA CLEANING

# Downloading the data, converting it to utf-8, and removing carriage return characters for proper handling of DOS Format:
!curl -s "https://opendata.dwd.de/climate_environment/CDC/helprr_wetter_tageswerte_Beschreibung_Stationen.txt" | iconv -f iso-8859-1 -t utf-8 | tr -d '\r' > stations.txt

STATES = ['Baden-Württemberg', 'Bayern', 'Berlin', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen']

# Converting .txt file to Pandas DataFrame for easier manipulation:
rows = []
with open('stations.txt', encoding='utf-8') as stations_f:
    for line in stations_f:
        line = line.strip()
        parts = line.split(None, 6)     # Splitting numerical entries by whitespace.
        if len(parts) > 6:
            remaining = parts[6]        # Remaining text after first 6 columns.
            # Finding the earliest occurrence of any state name in the remaining text:
            seventh_idx = -1
            for marker in STATES:
                idx = remaining.find(marker)
                if idx != -1 and (seventh_idx == -1 or idx < seventh_idx):
                    seventh_idx = idx
            # Splitting sixth from seventh column:
            if seventh_idx != -1:
                parts[6] = remaining[:seventh_idx].strip()
                state_text = remaining[seventh_idx:].strip()
                # Cleaning last column to contain nothing but state names:
                for state in STATES:
                    if state_text.startswith(state):
                        parts.append(state)
                        break
        rows.append(parts)
stations_df = pd.DataFrame(rows).set_index(0)   # Setting index.

# Adjusting header:
stations_df.columns = stations_df.iloc[0]
stations_df = stations_df.iloc[2:]
stations_df.columns.values[-1] = 'Bundesland'
stations_df.columns.values[-2] = 'Stationsname'
# Converting Pandas DataFrame to .csv file:
stations_df.to_csv('stations.csv', index=False)